# Weather RAG — Setup & Smoke Test
Run these cells on a Databricks cluster with the required libraries installed.
Ensure the `lakebase` secret (`database` scope, key `lakebase-url`) is set before running.

In [ ]:
# 1) Ensure weather tables (creates pgvector extension and tables)
import lakebase
lakebase.ensure_weather_tables(embedding_dim=384)
print('weather tables ensured')

In [ ]:
# 2) Sync a small location (example) to ingest raw weather documents
import weather_client
synced = weather_client.sync_locations(['San Francisco, CA'], limit=5)
print('documents synced:', synced)

In [ ]:
# 3) Run the embedding ingestion script on the cluster (this may install/require sentence-transformers and torch)
# Adjust the path below if your repo is mounted elsewhere in the workspace.
import os
script_path = '/Workspace/Repos/your-repo/scripts/ingest_weather_embeddings.py'
if os.path.exists(script_path):
    print('running ingestion script...')
    os.system(f'python 
 --states CA --model sentence-transformers/all-MiniLM-L6-v2')
else:
    print('Please update script_path to the correct repo path before running this cell')

In [ ]:
# 4) Quick search smoke test (embeddings must exist)
from sentence_transformers import SentenceTransformer
import lakebase
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
q = 'flash flood risk this weekend'
vec = model.encode([q], show_progress_bar=False)[0].tolist()
qvec = '[' + ','.join(map(lambda x: repr(float(x)), vec)) + ']'
rows = lakebase.run_query(
    'SELECT d.id AS document_id, d.source AS source, d.title AS headline, e.chunk_text, (e.embedding <=> %s::vector) AS distance FROM weather_embeddings e JOIN weather_documents d ON d.id = e.document_id ORDER BY distance ASC LIMIT %s',
    (qvec, 5)
)
print(rows)